In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from hls4ml_gravnet.utils.files import get_project_root_dir

PROJECT_ROOT = get_project_root_dir("hls4ml-gravnet")
RUN = "mini_128_L1_FP"
MEASUREMENTS = "gpu_measurements_L40S"

results_dir = f"{PROJECT_ROOT}/data/results/{RUN}/{MEASUREMENTS}"

In [ ]:
def load_trtexec_json_by_path(path: str | Path) -> pd.DataFrame:
    """
    Loads a trtexec per-iteration JSON file containing a list of dicts like:
      {"computeMs": ..., "latencyMs": ..., "h2dMs": ..., "d2hMs": ..., ...}
    Returns a DataFrame with numeric columns.
    """
    path = Path(path)
    with path.open("r") as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    for c in df.columns:
        if c.endswith("Ms") or c.startswith("start") or c.startswith("end"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

def load_trtexec_json(name: str) -> pd.DataFrame:
    """
    Loads the trtexec per-iteration JSON file for a given run.
    """
    path = Path(results_dir) / f"{name}.json"
    return load_trtexec_json_by_path(path)

def summarize_metric(df: pd.DataFrame, metric: str = "latencyMs", warmup: int = 0) -> pd.Series:
    """
    Summary stats for a metric column (default latencyMs).
    warmup: drop first N iterations to remove warm-up effects.
    """
    if warmup > 0:
        df = df.iloc[warmup:].reset_index(drop=True)

    x = df[metric].dropna().to_numpy()
    if x.size == 0:
        return pd.Series(dtype=float)

    return pd.Series({
        "n": x.size,
        "mean_ms": float(np.mean(x)),
        "std_ms": float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        "p50_ms": float(np.percentile(x, 50)),
        "p90_ms": float(np.percentile(x, 90)),
        "p95_ms": float(np.percentile(x, 95)),
        "p99_ms": float(np.percentile(x, 99)),
        "min_ms": float(np.min(x)),
        "max_ms": float(np.max(x)),
    })

def summary_df_across_measurements(names: list[str], metric: str = "latencyMs", warmup: int = 0) -> pd.DataFrame:
    """
    Summary stats for a metric column (default latencyMs) across multiple measurement files.
    Returns a DataFrame with one row per file and columns for the summary stats.
    """
    summaries = []
    for name in names:
        df = load_trtexec_json(name)
        summary = summarize_metric(df, metric=metric, warmup=warmup)
        summary["name"] = name
        summaries.append(summary)

    return pd.DataFrame(summaries).set_index("name")

In [ ]:
for file in Path(results_dir).iterdir():
    if file.is_file() and file.suffix == ".json" and "compute" in file.name:
        name = file.stem 
        df = load_trtexec_json(name)
        summary = summarize_metric(df, metric="latencyMs", warmup=10)
        print(f"{name}: {summary['mean_ms']:.2f} ms (n={summary['n']})")
    